# Experiment 2.1b: Does Conjugation Covariance Persist Through Momentum?

## Counterpart identity and calibrated scope

This notebook is the narrative/figure counterpart to
`run_experiment.py` in the same experiment directory. It deliberately **imports and
runs the script's reusable `run_experiment()` function** instead of re-implementing
its core logic, so the notebook and script remain behaviorally aligned.

### What this pair actually studies in this first completion pass

We keep the original three-part single-matrix study under bilateral orthogonal
conjugation
\[
W \mapsto RWS^\top,
\]
using Muon-style updates with momentum:

1. **A. Random conjugated gradients** — positive control.
2. **B. Fixed-frame single-layer linear MSE gradients** — negative control.
3. **C. Equivariant Frobenius-loss gradients** — positive control.

### What is measured

- **Primary metric:** final relative covariance error after the configured number of
  updates,
  \[
  \frac{\lVert W_T' - R W_T S^\top \rVert_F}{\lVert W_T \rVert_F}.
  \]
- **Secondary diagnostics:** sampled single-trial trajectories across the 50 update
  steps, including the data-driven loss traces for the negative control.

### What is *not* claimed here

This notebook does **not** claim unconditional or universal behavior across all
matrix sizes, all momentum values, all losses, or all architectures. In this first
pass, the main empirical message is narrower:

- in the tested float64 setup, the two positive controls remain at machine precision;
- the fixed-frame data-driven case fails badly;
- that failure is attributed here to the **non-equivariant fixed-frame gradient map**
  under this setup, not to momentum in isolation.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import platform
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None


def render_and_close(fig):
    if display is not None:
        display(fig)
    else:
        fig.canvas.draw()
    plt.close(fig)


## Notebook-safe path resolution and script import

The notebook avoids `__file__` and does not assume a fragile current working
directory. Instead, it walks upward from `Path.cwd()` until it finds the repository
root containing the expected experiment script, then imports that script by path.


In [ ]:
REPO_NAME = 'Muon_as_RG_Gauge_Fixing'
SCRIPT_RELATIVE_PATH = Path(
    'experiments'
) / 'Tier_2_Symmetry_Reparametrization_Tests' / 'Exp_2.1_Conjugation_Covariance' / \
    '2.1b_Conjugation_Does_Covariance_Persist_Through_Momentum' / 'run_experiment.py'


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / SCRIPT_RELATIVE_PATH).exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate repository root by walking upward from the current working directory. '
        f'Expected to find {SCRIPT_RELATIVE_PATH} in an ancestor of {start}.'
    )


repo_root = find_repo_root()
script_path = repo_root / SCRIPT_RELATIVE_PATH
module_name = 'exp_2_1b_run_experiment'
spec = importlib.util.spec_from_file_location(module_name, script_path)
if spec is None or spec.loader is None:
    raise ImportError(f'Could not create import spec for {script_path}')
experiment_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = experiment_module
spec.loader.exec_module(experiment_module)

print(f'Repository root: {repo_root}')
print(f'Imported script: {script_path}')
print(f'Imported module name: {experiment_module.__name__}')


## Reproducibility, runtime, and configuration

The code cell below executes the same default experiment suite used by the script
entrypoint. This provides a single source of truth for the notebook's tables,
figures, and textual conclusions.


In [ ]:
results = experiment_module.run_experiment()
config = results['config']
experiment_results = results['experiment_results']
checks = results['heuristic_checks']
diagnostics = results['single_trial_diagnostics']
EXPERIMENT_ORDER = list(experiment_module.EXPERIMENT_ORDER)
HEURISTIC_CHECK_ORDER = list(experiment_module.HEURISTIC_CHECK_ORDER)

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Matplotlib:', plt.matplotlib.__version__)
print(f"Script-reported runtime: {results['runtime_seconds']:.3f} s")
print('Scope note:')
print(results['scope_note'])
print('Configuration:')
print(json.dumps(config, indent=2))


### Interpretation

This is a deliberately lightweight CPU experiment: no deep learning framework or
GPU is required, and the default run should complete in roughly the low-second
range. The point of the notebook is not to change the experiment, but to make the
existing study transparent, reproducible, and easier to interpret.


In [ ]:
summary_rows = []
raw_rows = []
for experiment_key in EXPERIMENT_ORDER:
    block = experiment_results[experiment_key]
    for size_label, size_result in block['sizes'].items():
        summary = size_result['summary']
        summary_rows.append({
            'experiment_key': experiment_key,
            'experiment': block['label'],
            'size': size_label,
            'n': summary['n'],
            'mean': summary['mean'],
            'std': summary['std'],
            'median': summary['median'],
            'min': summary['min'],
            'max': summary['max'],
            'ci95_low': summary['ci95_low'],
            'ci95_high': summary['ci95_high'],
        })
        for seed, err in zip(size_result['trial_seeds'], size_result['raw_trial_errors']):
            raw_rows.append({
                'experiment_key': experiment_key,
                'experiment': block['label'],
                'size': size_label,
                'seed': seed,
                'final_error': err,
            })

summary_df = pd.DataFrame(summary_rows)
raw_df = pd.DataFrame(raw_rows)
comparison_df = pd.DataFrame(results['comparison_rows'])

print('Summary table (final relative covariance error):')
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.3e}'))
print('\nComparison rows (means by size):')
print(comparison_df.to_string(index=False, float_format=lambda x: f'{x:.3e}'))


## Final-endpoint summary across the three sub-experiments

The table above is the main numerical backbone of the study. It gives the raw
descriptive statistics actually implemented in the script: means, spreads,
medians, extrema, and simple normal-approximation 95% confidence intervals for the
trial-wise final covariance error.

### How to read it

- **A** and **C** are the positive controls. Their final errors should stay near
  floating-point machine precision if covariance is preserved numerically.
- **B** is the negative control. Its error should be macroscopically larger because
  the fixed-frame gradient map does not commute with the conjugation action in this
  setup.


In [ ]:
size_labels = [f"{m}x{n}" for m, n in config['matrix_sizes']]
fig, axes = plt.subplots(1, len(size_labels), figsize=(6 * len(size_labels), 4), sharey=True)
if len(size_labels) == 1:
    axes = [axes]

for ax, size_label in zip(axes, size_labels):
    subset = raw_df[raw_df['size'] == size_label]
    series = [
        subset[subset['experiment_key'] == key]['final_error'].to_numpy()
        for key in EXPERIMENT_ORDER
    ]
    labels = ['A', 'B', 'C']
    ax.boxplot(series, tick_labels=labels, showfliers=True)
    for xpos, values, color in zip(range(1, 4), series, ['tab:blue', 'tab:orange', 'tab:green']):
        offsets = np.linspace(-0.08, 0.08, len(values)) if len(values) > 1 else np.array([0.0])
        ax.scatter(np.full(len(values), xpos) + offsets, values, s=18, alpha=0.7, color=color)
    ax.set_yscale('log')
    ax.set_title(f'Final errors by condition ({size_label})')
    ax.set_xlabel('Condition')
    ax.grid(True, which='both', alpha=0.3)

axes[0].set_ylabel('Final relative covariance error (log scale)')
fig.suptitle('Distribution of final covariance errors across seeds', y=1.02)
fig.tight_layout()
render_and_close(fig)


### Interpretation of the final-error distributions

The distributional plot makes the contrast visually explicit:

- the two positive controls sit near machine precision across all tested seeds;
- the data-driven negative control is many orders of magnitude larger;
- the observed separation is strong enough that the ratio heuristics later in the
  notebook are unsurprising, even though they are still only heuristics.


In [ ]:
trajectory_block = diagnostics['data_vs_equivariant']
random_block = diagnostics['random_gradient']
trajectory_df = pd.DataFrame({
    'step': trajectory_block['step_indices'],
    'A_random_conjugated_gradients': random_block['error_trace'],
    'B_fixed_frame_data_driven': trajectory_block['data_driven_error_trace'],
    'C_equivariant_frobenius_loss': trajectory_block['equivariant_loss_error_trace'],
})

sampled_df = pd.DataFrame(trajectory_block['sampled_rows'])
random_sampled_df = pd.DataFrame(random_block['sampled_rows'])

print('Sampled single-trial trajectory rows (data-driven vs equivariant-loss):')
print(sampled_df.to_string(index=False, float_format=lambda x: f'{x:.3e}'))
print('\nSampled single-trial trajectory rows (random conjugated gradients):')
print(random_sampled_df.to_string(index=False, float_format=lambda x: f'{x:.3e}'))

fig, ax = plt.subplots(figsize=(8, 4.8))
epsilon = 1e-20
ax.plot(trajectory_df['step'], np.maximum(trajectory_df['A_random_conjugated_gradients'], epsilon), label='A: random conjugated gradients', lw=2)
ax.plot(trajectory_df['step'], np.maximum(trajectory_df['B_fixed_frame_data_driven'], epsilon), label='B: fixed-frame data-driven', lw=2)
ax.plot(trajectory_df['step'], np.maximum(trajectory_df['C_equivariant_frobenius_loss'], epsilon), label='C: equivariant Frobenius loss', lw=2)
ax.set_yscale('log')
ax.set_xlabel('Optimization step')
ax.set_ylabel('Relative covariance error (log scale)')
ax.set_title(
    'Single-trial covariance trajectories '
    f"({trajectory_block['size'][0]}x{trajectory_block['size'][1]}, seed={trajectory_block['seed']})"
)
ax.grid(True, which='both', alpha=0.3)
ax.legend()
fig.tight_layout()
render_and_close(fig)


## Trajectory interpretation

This trajectory view is important because the script does not only report final
errors; it also computes a representative single-trial path through optimization.
Here the pattern is clear:

- **A** and **C** remain near machine precision throughout the sampled trajectory.
- **B** quickly departs from the conjugation-related path and stays macroscopically
  separated.

That is still not a proof about *every* run or *every* momentum setting, but it is
strong within the sampled toy regime implemented here.


In [ ]:
loss_df = pd.DataFrame({
    'step': trajectory_block['step_indices'],
    'path_a_loss': trajectory_block['data_driven_loss_trace_path_a'],
    'path_b_loss': trajectory_block['data_driven_loss_trace_path_b'],
})

print('Data-driven loss-trace endpoints:')
print(loss_df.iloc[[0, -1]].to_string(index=False, float_format=lambda x: f'{x:.3e}'))
print('\nInitial/final path-loss gap metadata:')
print(json.dumps(trajectory_block['metadata'], indent=2))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(loss_df['step'], loss_df['path_a_loss'], label='Path A loss', lw=2)
ax.plot(loss_df['step'], loss_df['path_b_loss'], label='Path B loss', lw=2)
ax.set_xlabel('Optimization step')
ax.set_ylabel('Single-layer MSE loss')
ax.set_title('Data-driven negative control: loss traces for the two paths')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
render_and_close(fig)


### Why the data-driven case fails here

The negative control does not fail because “momentum generically destroys
symmetry.” The implementation is more specific:

- both paths are trained against the **same fixed-frame** inputs and targets;
- under that setup, the single-layer MSE gradient map is not conjugation-equivariant;
- once the gradient map itself fails to commute with `W -> R W S^T`, the optimizer
  no longer has a covariance relation to preserve.

The loss plot is therefore complementary evidence: optimization can proceed in both
paths while the conjugation relationship between the paths is nonetheless broken.


## Heuristic threshold checks (not formal hypothesis tests)

The original study used four threshold checks H1–H4. In this notebook they are kept
for continuity, but they are presented explicitly as **heuristic checks**, not as
formal statistical tests.


In [ ]:
check_rows = []
for key in HEURISTIC_CHECK_ORDER:
    item = checks[key]
    row = {
        'check': key,
        'statement': item['statement'],
        'criterion': item['criterion'],
        'pass': item['pass'],
    }
    row.update(item['observed'])
    check_rows.append(row)

checks_df = pd.DataFrame(check_rows)
print(checks_df.to_string(index=False, float_format=lambda x: f'{x:.3e}'))

passed = checks['checks_passed']
total = checks['checks_total']
print(f'\nHeuristic checks passed: {passed}/{total}')


### Interpretation of H1–H4

- **H1** and **H3** ask whether the positive controls stay below a stringent
  machine-precision-style threshold at the tested final endpoints.
- **H2** asks whether the negative control is macroscopically broken.
- **H4** is only a ratio heuristic. It is useful as a scale comparison, but it is
  not a proof and should not be over-interpreted.


In [ ]:
means = checks['aggregate_means']
maxima = checks['aggregate_maxima']
ratio_ba = checks['H4']['observed']['ratio_B_over_A']
ratio_bc = checks['H4']['observed']['ratio_B_over_C']

print('Calibrated conclusion with observed values:')
print('- Positive control A mean / max final error: '
      f"{means['A_random_conjugated_gradients']:.3e} / {maxima['A_random_conjugated_gradients']:.3e}")
print('- Negative control B mean / max final error: '
      f"{means['B_fixed_frame_data_driven']:.3e} / {maxima['B_fixed_frame_data_driven']:.3e}")
print('- Positive control C mean / max final error: '
      f"{means['C_equivariant_frobenius_loss']:.3e} / {maxima['C_equivariant_frobenius_loss']:.3e}")
print(f'- Heuristic ratio B/A: {ratio_ba:.3e}')
print(f'- Heuristic ratio B/C: {ratio_bc:.3e}')


## Final conclusion

In this first pair-level completion pass, the notebook now matches the script's core
behavior while making the evidence much easier to audit.

### Empirical conclusion within the sampled numerical scope

At the tested float64 configuration:

- the **random conjugated-gradient** and **equivariant Frobenius-loss** controls stay
  at machine precision in their final errors and representative trajectories;
- the **fixed-frame single-layer data-driven** control develops large covariance
  error under the same Muon+momentum update rule.

### Interpretation

The natural interpretation of this pair of controls is not that momentum alone is
responsible for symmetry failure. Rather, in this setup the observed failure occurs
when the **gradient map itself is not conjugation-equivariant**.

### Remaining limits of this notebook

This notebook still does not prove universal statements about all step counts, all
betas, all losses, all matrix sizes, or multi-layer gauge-covariant training. Those
would require either additional experiments or separate theoretical arguments.
